# Device Safety

In PyTorch, forgetting `.to(device)` is a runtime crash:

```python
model = Net()           # CPU
x = torch.randn(10).cuda()  # GPU
model(x)                # RuntimeError: expected cpu but got cuda:0
```

In idris-ml, `Variable` carries a phantom `Device` parameter. The **compiler**
catches device mismatches — no runtime crash, no debugging, no silent wrong results.


## The Device Type

Three devices are supported: CPU, CUDA (with GPU index), and MPS (Apple Metal).


In [1]:
:browse Device


CPU : Device
CUDA : Nat -> Device
Device : Type
MPS : Device
deviceToString : Device -> String


## Variable Carries Its Device

`Variable` has an erased device parameter: `record Variable (0 d : Device)`.

The `0` means it exists only at compile time — zero runtime cost.


In [2]:
:t MkTensor


Variable.Var : AnyPtr -> Maybe String -> Double -> Variable d


The constructor `Var : AnyPtr -> Maybe String -> Double -> Variable d` creates a
variable on device `d`. Since `d` is erased, there's no device field at runtime —
the actual device lives in the C-side tensor.


## CPU and CUDA Are Different Types

`Variable CPU` and `Variable (CUDA 0)` are distinct types. You cannot mix them:


The following snippet **would fail to compile** because the model has device `CPU` but the input claims device `CUDA 0`:

```idris
:exec do { srand 42;
  ll <- linearLayerAny {i=2} {o=3} "ll0";
  model <- pure ((OutputLayer ll));  -- Network 2 [] 3 CPU
  let inT = the (Tensor [2] (CUDA 0)) (MkTensor (bulkToTensor (VArray [1.0, 2.0])) Nothing);
  let (_, outT) = forwardVar model inT;  -- compile error: device mismatch
  pure () }
```

Idris reports  at compile time. Use  for explicit transfer between device types.

The compiler rejects this with a clear error: `Mismatch between: CPU and CUDA 0`.

In PyTorch, this same operation would compile fine and crash at runtime.


## Networks Inherit the Device

When you build a network with `Variable CPU` layers, the entire network is typed
as `Network i hs o (Variable CPU)`. It can only accept `Variable CPU` inputs.


In [ ]:
:exec do { srand 42;
  ll <- linearLayerAny {i=2} {o=3} "ll0";
  putStrLn "Model OK" }


Trying to forward a CUDA vector through a CPU network is a compile error:


The following snippet **would fail to compile** because the model has device `CPU` but the input claims device `CUDA 0`:

```idris
:exec do { srand 42;
  ll <- linearLayerAny {i=2} {o=3} "ll0";
  model <- pure ((OutputLayer ll));  -- Network 2 [] 3 CPU
  let inT = the (Tensor [2] (CUDA 0)) (MkTensor (bulkToTensor (VArray [1.0, 2.0])) Nothing);
  let (_, outT) = forwardVar model inT;  -- compile error: device mismatch
  pure () }
```

Idris reports  at compile time. Use  for explicit transfer between device types.

Compare with PyTorch where this would be:
```python
model = nn.Linear(2, 3)          # CPU
x = torch.tensor([1.0, 2.0]).cuda()  # GPU
model(x)  # RuntimeError at line 3, not caught until you run the code
```

In idris-ml, the error is caught before the program ever runs.


## `toDevice`: The Intentional Bridge

When you genuinely need to move data between devices, `toDevice` makes it explicit:


In [6]:
:t toDevice


Variable.toDevice : (d2 : Device) -> Variable d1 -> IO (Variable d2)


`toDevice` takes a runtime `Device` argument (not erased) because it performs a
physical data transfer. This is the **only** way to change a variable's device type —
every other operation preserves it.

On the tape backend (CPU only), `toDevice` is a no-op.


## Zero Runtime Cost

The device parameter is quantity `0` — fully erased by the compiler. A `Variable CPU`
and a hypothetical `Variable (CUDA 0)` have the exact same runtime representation.
The safety is purely at compile time, like Rust's lifetime annotations.


In [7]:
:t fromDouble


Builtin.fromDouble : FromDouble ty => Double -> ty


`fromDouble` creates `Variable d` for any device `d` — the actual tensor lands on
whichever device the C backend defaults to. Currently all backends default to CPU.


## Summary

| | PyTorch | idris-ml |
|---|---------|----------|
| Device tracking | Runtime (`.device` property) | Compile-time (phantom type) |
| Mismatch detection | `RuntimeError` during forward pass | Type error before compilation |
| Cost | 0 (property lookup) | 0 (erased at compile time) |
| Device transfer | `.to(device)` | `toDevice device var` |
| Multi-device | Same `Tensor` type for all | `Variable CPU` and `Variable (CUDA 0)` are distinct types |

Next: back to [01 Tensors and Types](01_tensors_and_types.ipynb) for the basics.
